# Phân tích nâng cao — Vòng đời & dịch chuyển khách hàng Olist

Notebook này mở rộng phân tích phân khúc RFM sang hai câu hỏi sâu hơn về hành vi khách hàng
theo thời gian, nhằm chuyển từ mô tả ("khách thuộc nhóm nào") sang định lượng có thể hành động
("khi nào một khách được coi là đã rời bỏ", "khách dịch chuyển giữa các nhóm ra sao").

**Nội dung:**

1. Phân tích sống sót (Survival Analysis) — xác định ngưỡng churn dựa trên dữ liệu thay vì
   chọn tùy ý, thông qua phân phối thời gian giữa các lần mua.
2. Dịch chuyển phân khúc (Segment Migration) — so sánh phân khúc khách hàng giữa hai mốc thời
   gian để đo dòng chảy giữa các nhóm.
3. Mô phỏng business case — ước lượng giá trị doanh thu có thể thu hồi từ các chiến dịch giữ chân.
4. Thiết kế A/B test — khung kiểm chứng hiệu quả chiến dịch trước khi triển khai thực tế.

**Nguồn dữ liệu:** `gold.fact_orders` (data warehouse PostgreSQL).

## 1. Phân tích sống sót (Survival Analysis)

**Mục tiêu:** Xác định ngưỡng thời gian hợp lý để coi một khách hàng là đã rời bỏ (churn),
dựa trên bằng chứng dữ liệu thay vì con số chọn tùy ý.

**Phương pháp:** Phân tích sống sót nghiên cứu "thời gian đến khi một sự kiện xảy ra". Ở đây
sự kiện là "khách quay lại mua lần tiếp theo", và biến quan tâm là khoảng cách ngày giữa hai
đơn hàng liên tiếp (inter-purchase time).

**Lưu ý về kiểm duyệt phải (right censoring):** 97% khách chỉ mua một lần. Với họ, thời gian
đến lần mua kế tiếp là *chưa quan sát được* — không phải bằng 0 hay vô hạn, mà bị cắt cụt tại
thời điểm dữ liệu kết thúc (10/2018). Vì vậy không thể tính trung bình thông thường (sẽ thiên
lệch nặng do bỏ qua nhóm chưa quay lại). Phần 1.1 phân tích trên nhóm khách đã mua lặp lại để
hiểu hành vi quay lại; phần 1.2 thảo luận giới hạn của cách tiếp cận này.

### 1.1. Chuẩn bị dữ liệu

Trích xuất mỗi đơn hàng một dòng kèm mã khách và ngày mua, sắp xếp theo khách và thời gian —
làm cơ sở tính khoảng cách giữa các đơn liên tiếp.

In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
from sqlalchemy import create_engine 
from dotenv import load_dotenv
import os 

load_dotenv()
db_url = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(db_url)

df = pd.read_sql(
"""
SELECT customer_unique_id , order_id , order_purchase_date
FROM gold.fact_orders
ORDER BY customer_unique_id , order_purchase_date
""",engine)

print(df.shape)
df.head()

### 1.2. Tính khoảng cách giữa các lần mua

Với mỗi khách hàng, sắp xếp đơn theo thời gian và tính số ngày giữa các đơn liên tiếp. Ở phân
tích này tập trung vào khoảng cách từ đơn **thứ nhất đến đơn thứ hai** — đây là bước chuyển đổi
quan trọng nhất: một khách mua lần đầu có quay lại lần hai hay không quyết định phần lớn giá trị
vòng đời của họ.

In [ ]:
# Đảm bảo kiểu ngày và sắp xếp
df['order_purchase_date'] = pd.to_datetime(df['order_purchase_date'])
df = df.sort_values(['customer_unique_id', 'order_purchase_date'])

# Số thứ tự đơn của mỗi khách (1, 2, 3...) và ngày đơn kế tiếp
df['order_seq'] = df.groupby('customer_unique_id').cumcount() + 1
df['next_date'] = df.groupby('customer_unique_id')['order_purchase_date'].shift(-1)

# Khoảng cách đến đơn kế tiếp (ngày). Đơn cuối của mỗi khách -> NaT (chưa có đơn sau)
df['days_to_next'] = (df['next_date'] - df['order_purchase_date']).dt.days

# Chỉ lấy khoảng cách đơn 1 -> đơn 2, và chỉ những khách THỰC SỰ có đơn 2
gap_1_to_2 = df[(df['order_seq'] == 1) & (df['days_to_next'].notna())]['days_to_next']

print(f"Số khách có đơn thứ 2: {len(gap_1_to_2):,}")
print(gap_1_to_2.describe())

#### Kiểm tra: khoảng cách 0 ngày

Phân vị 25% bằng 0 cho thấy một tỷ lệ đáng kể đơn thứ hai phát sinh cùng ngày với đơn thứ nhất.
Cần xác minh đây có phải hành vi mua lại thực sự, hay là một lần mua bị tách thành nhiều đơn

Nếu các đơn cùng ngày thực sự là split orders (một lần mua bị tách), chúng phải phát sinh cách
nhau chỉ vài giây — cùng một phiên bấm mua. Ta kiểm ở mức timestamp (giây), truy xuất từ
`olist_orders` (bronze) vì lớp gold đã làm tròn ngày.

In [ ]:
# Query timestamp đầy đủ từ bronze
orders_ts = pd.read_sql("""
    SELECT c.customer_unique_id, o.order_purchase_timestamp
    FROM olist_orders o
    JOIN olist_customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    ORDER BY c.customer_unique_id, o.order_purchase_timestamp
""", engine)

orders_ts['order_purchase_timestamp'] = pd.to_datetime(orders_ts['order_purchase_timestamp'])

# Khoảng cách (giây) tới đơn liền trước của cùng khách
orders_ts['prev_ts'] = orders_ts.groupby('customer_unique_id')['order_purchase_timestamp'].shift(1)

# Chỉ giữ các cặp đơn CÙNG NGÀY
same_day = orders_ts[
    orders_ts['prev_ts'].dt.date == orders_ts['order_purchase_timestamp'].dt.date
].copy()
same_day['gap_seconds'] = (
    same_day['order_purchase_timestamp'] - same_day['prev_ts']
).dt.total_seconds()

g = same_day['gap_seconds']
print(f"Số cặp đơn cùng khách cùng ngày: {len(same_day)}")
print(f"Median khoảng cách:  {g.median():.0f} giây")
print(f"<= 60 giây:  {(g <= 60).mean()*100:.1f}%")
print(f"<= 1 giờ :    {(g <= 3600 ).mean()*100:.1f}%")
print(f"> 6 giờ:     {(g > 21600).mean()*100:.1f}%")

**Nhận xét & quyết định xử lý:** Median khoảng cách giữa hai đơn cùng ngày chỉ **1 giây** (81%
dưới 60 giây, 96% trong vòng một giờ, chỉ 1.5% quá 6 giờ). Khoảng cách một giây loại trừ hoàn
toàn khả năng khách quay lại mua lần hai — đây là **một lần mua bị tách thành nhiều order_id**,
do khách mua từ nhiều người bán trong cùng giỏ hàng và Olist tách đơn theo người bán.

→ **Vì bằng chứng này**, ta định nghĩa lại đơn vị phân tích: một "lần mua" (purchase occasion)
là một **ngày mua riêng biệt** của khách, gộp mọi đơn cùng ngày thành một. Phần tiếp theo tính
lại khoảng cách quay lại trên đơn vị đã hiệu chỉnh này — đó mới là thước đo hành vi mua lại thực
sự, làm nền cho việc xác định ngưỡng churn.

#### Định nghĩa lại "lần mua" (purchase occasion)



In [ ]:
# Gộp về ngày mua riêng biệt (1 lần mua = 1 ngày), loại trùng đơn cùng ngày
occ = df[['customer_unique_id', 'order_purchase_date']].drop_duplicates()
occ = occ.sort_values(['customer_unique_id', 'order_purchase_date'])

# Tính lại khoảng cách giữa các LẦN MUA thật
occ['seq'] = occ.groupby('customer_unique_id').cumcount() + 1
occ['next_date'] = occ.groupby('customer_unique_id')['order_purchase_date'].shift(-1)
occ['days_to_next'] = (occ['next_date'] - occ['order_purchase_date']).dt.days

gap_clean = occ[(occ['seq'] == 1) & (occ['days_to_next'].notna())]['days_to_next']

print(f"Số khách thực sự mua lại (khác ngày): {len(gap_clean):,}")
print(f"So với đếm theo order_id: 2,801 → {len(gap_clean):,}")
print(gap_clean.describe())

### 1.3. Phân phối tích lũy (CDF) và ngưỡng churn

Trên nhóm 2,015 khách thực sự mua lại, đường CDF cho biết "bao
nhiêu % khách quay lại đã quay lại trong vòng Y ngày". Ngưỡng churn hợp lý là điểm mà phần lớn
khách-sẽ-quay-lại đã quay lại — vượt mốc đó, xác suất quay lại rất thấp nên coi như đã rời bỏ.
Xét các phân vị cao (P80, P90, P95) làm ứng viên thay cho ngưỡng 90 ngày chọn tùy ý ban đầu.

In [ ]:
for p in [75, 80, 90, 95]:
    print(f"P{p}: {gap_clean.quantile(p/100):.0f} ngày")

sorted_gap = np.sort(gap_clean)
cdf = np.arange(1, len(sorted_gap) + 1) / len(sorted_gap)

plt.figure(figsize=(10, 5))
plt.plot(sorted_gap, cdf * 100, color='seagreen')
plt.axhline(90, color='red', linestyle='--', alpha=0.6, label='90% khách quay lại')
plt.xlabel('Số ngày từ lần mua 1 đến lần mua 2')
plt.ylabel('% khách tích lũy đã quay lại')
plt.title('CDF: Thời gian quay lại mua lần hai ')
plt.legend()
plt.grid(alpha=.3)
plt.show()

#### So sánh với quy ước "90-day churn window"

Trong phân tích khách hàng, một quy ước phổ biến là coi khách "đã rời bỏ" (churned) nếu không
phát sinh giao dịch trong **90 ngày**. Ngưỡng này được nhiều doanh nghiệp và công cụ CRM dùng
làm mặc định vì đơn giản và dễ áp dụng, nhưng nó là con số kinh nghiệm chung — không xuất phát
từ hành vi thực tế của tập khách hàng cụ thể nào.

Ta kiểm chứng xem ngưỡng 90 ngày có phù hợp với dữ liệu Olist không, bằng cách xem tại mốc 90
ngày đã có bao nhiêu phần trăm khách quay lại.

In [ ]:
pct_90 = (gap_clean <= 90).mean() * 100
print(f"Tại mốc 90 ngày: {pct_90:.0f}% khách quay lại đã quay lại")
print(f"Còn lại {100-pct_90:.0f}% khách vẫn sẽ quay lại nhưng bị ngưỡng 90 ngày coi là 'đã mất'")

**Nhận xét:** Tại mốc 90 ngày, chỉ khoảng 60% khách quay lại đã thực sự quay lại. Nếu áp dụng
ngưỡng churn 90 ngày cho Olist, khoảng 40% khách vẫn còn khả năng quay lại sẽ bị phân loại nhầm
là "đã mất" — dẫn tới đánh giá sai quy mô churn và có thể lãng phí ngân sách win-back cho nhóm
chưa thực sự rời bỏ.

Đây là lý do ngưỡng churn nên được xác định từ dữ liệu thay vì áp dụng quy ước chung. Với Olist,
phân tích đề xuất hai ngưỡng phục vụ hai mục đích:

| Ngưỡng | Giá trị | Ý nghĩa & cách dùng |
| :-- | :-- | :-- |
| Cần can thiệp (about-to-sleep) | ~74–175 ngày (median → P75) | Thời điểm nên kích hoạt chiến dịch giữ chân, khi khách bắt đầu trôi xa nhưng chưa mất |
| Đã rời bỏ (churned) | ~288 ngày (P90) | Ngưỡng phân loại khách đã mất hẳn; qua mốc này gần như không quay lại |

## 2. Dịch chuyển vòng đời khách hàng (Customer Lifecycle Migration)

**Mục tiêu:** Đo dòng chảy của khách hàng giữa các trạng thái vòng đời theo thời gian, để chuyển
từ ảnh chụp tĩnh ("hiện khách thuộc nhóm nào") sang xu hướng động ("khách đang dịch chuyển ra
sao"). Đây là thông tin ban lãnh đạo cần để đánh giá sức khỏe tệp khách hàng.

**Thiết kế:**
- So sánh hai thời điểm quan sát: kỳ 1 chốt 28/02/2018, kỳ 2 chốt 30/08/2018 (cách nhau 6 tháng).
  Tại mỗi mốc, RFM được tính từ toàn bộ đơn phát sinh *đến* thời điểm đó.
- **Trạng thái vòng đời** được gán bằng ngưỡng recency cố định. Đây là một bộ phân loại đơn giản
  hơn 5 phân khúc K-Means ở phần trước, phục vụ riêng cho mục đích đo dịch chuyển: dùng ngưỡng cố
  định (thay vì phân cụm lại từng kỳ) đảm bảo cùng một thước đo cho cả hai kỳ, nên dịch chuyển đo
  được là thật chứ không phải do thang đo thay đổi. Đặt tên riêng (Engaged/Cooling/Dormant) để
  không nhầm lẫn với nhãn phân khúc RFM.

  Hai ngưỡng 175 và 288 chính là hai ngưỡng đã xác định từ survival analysis ở phần 1 (P75 và
  P90) — tái sử dụng để giữ tính liền mạch giữa hai phân tích:

  | Trạng thái | Điều kiện | Ý nghĩa |
  | :-- | :-- | :-- |
  | Engaged | recency ≤ 175 ngày | Còn tương tác, chưa cần can thiệp |
  | Cooling | 175 < recency ≤ 288 ngày | Đang nguội — vùng nên can thiệp |
  | Dormant | recency > 288 ngày | Đã rời bỏ |

- **Monetary** không dùng để phân nhóm mà làm trọng số: mỗi luồng dịch chuyển được đo bằng cả số
  khách lẫn tổng doanh thu cuốn theo.

### 2.1. Hàm tính trạng thái vòng đời tại một thời điểm

Xây một hàm tái sử dụng: cho một mốc thời gian, hàm tính RFM từ toàn bộ đơn phát sinh *đến* mốc
đó (không dùng dữ liệu tương lai để tránh rò rỉ), rồi gán trạng thái vòng đời theo ngưỡng recency
cố định. Dùng chung một hàm cho cả hai kỳ đảm bảo hai kỳ được đo bằng cùng một cách — điều kiện
để so sánh dịch chuyển có ý nghĩa.

In [1]:
def compute_lifecycle(df, snapshot_date):
    """Tính recency & gán trạng thái vòng đời cho các khách có đơn TRƯỚC snapshot_date."""
    snapshot = pd.Timestamp(snapshot_date)
    # Chỉ nhìn đơn đã xảy ra tính đến mốc (không dùng dữ liệu tương lai)
    hist = df[df['order_purchase_date'] <= snapshot]

    g = hist.groupby('customer_unique_id').agg(
        recency=('order_purchase_date', lambda x: (snapshot - x.max()).days),
        frequency=('order_id', 'nunique'),
        monetary=('total_payment_value', 'sum') if 'total_payment_value' in df.columns else ('order_id','size')
    ).reset_index()

    # Gán trạng thái vòng đời bằng ngưỡng cố định (cùng thước cho mọi kỳ)
    def stage(r):
        if r <= 175: return 'Engaged'
        elif r <= 288: return 'Cooling'
        else: return 'Dormant'
    g['lifecycle'] = g['recency'].apply(stage)
    return g

Error: No connection selected.

### 2.2. Áp dụng cho hai kỳ và kiểm tra phân bố

Truy xuất dữ liệu đơn hàng (kèm giá trị thanh toán để làm trọng số), rồi áp hàm cho hai mốc:
28/02/2018 và 30/08/2018. Kiểm tra phân bố trạng thái ở mỗi kỳ. Kỳ 1 dự kiến có ít khách hơn kỳ 2
vì chốt sớm hơn — nhiều khách chưa phát sinh đơn nào tại thời điểm đó.

In [2]:
df = pd.read_sql("""
    SELECT customer_unique_id, order_id, order_purchase_date, total_payment_value
    FROM gold.fact_orders
    ORDER BY customer_unique_id, order_purchase_date
""", engine)
df['order_purchase_date'] = pd.to_datetime(df['order_purchase_date'])

# Áp hàm cho 2 kỳ
p1 = compute_lifecycle(df, '2018-02-28')
p2 = compute_lifecycle(df, '2018-08-30')

print(f"Kỳ 1 (28/02/2018): {len(p1):,} khách")
print(f"Kỳ 2 (30/08/2018): {len(p2):,} khách")
print("\nPhân bố trạng thái kỳ 1:")
print(p1['lifecycle'].value_counts())
print("\nPhân bố trạng thái kỳ 2:")
print(p2['lifecycle'].value_counts())

Error: No connection selected.

**Nhận xét:** Kỳ 1 có 55,524 khách (ít hơn kỳ 2 vì nhiều khách chưa xuất hiện tại 28/02/2018),
kỳ 2 có đủ 93,357 khách 

Không so sánh trực tiếp hai phân bố tổng thể này, vì hai tập khách khác nhau (kỳ 2 có thêm ~37,800
khách mới). Phân bố tổng chỉ cho bối cảnh; để đo dịch chuyển thực sự cần theo dõi cùng một tập
khách qua hai kỳ — thực hiện ở ma trận chuyển tiếp phần sau.

### 2.3. Ma trận chuyển tiếp (Transition Matrix)

Ghép trạng thái của từng khách ở hai kỳ để đo dòng chảy. Chỉ những khách có mặt ở **cả hai kỳ**
mới vào ma trận (khách chỉ xuất hiện ở kỳ 2 được xếp riêng là "New"). Mỗi ô là số khách đi từ
trạng thái hàng (kỳ 1) sang trạng thái cột (kỳ 2).

In [3]:
# Ghép trạng thái 2 kỳ theo khách. Chỉ giữ khách có mặt ở CẢ 2 kỳ (inner join)
merged = p1[['customer_unique_id', 'lifecycle']].merge(
    p2[['customer_unique_id', 'lifecycle', 'monetary']],
    on='customer_unique_id', suffixes=('_k1', '_k2')
)

print(f"Số khách có mặt ở cả 2 kỳ: {len(merged):,}")

# Ma trận đếm khách: hàng = trạng thái kỳ 1, cột = trạng thái kỳ 2
order = ['Engaged', 'Cooling', 'Dormant']
transition = pd.crosstab(merged['lifecycle_k1'], merged['lifecycle_k2']).reindex(index=order, columns=order)
print("\nMa trận chuyển tiếp (số khách):")
print(transition)

# Cùng ma trận nhưng theo % mỗi hàng (trong nhóm kỳ 1, bao nhiêu % đi về đâu)
print("\nMa trận chuyển tiếp (% theo hàng):")
print((transition.div(transition.sum(axis=1), axis=0) * 100).round(1))

Error: No connection selected.

**Nhận xét:** Khoảng cách hai kỳ (183 ngày) xấp xỉ ngưỡng Engaged
(175 ngày), nên một khách Engaged không mua lại sẽ tự động rời nhóm chỉ do thời gian trôi. Điều
này khiến ma trận khó tách bạch "dịch chuyển do thời gian trôi" khỏi "thay đổi hành vi thực sự" —
trên thực tế, "ở lại hoặc đi lên" gần như tương đương "có mua lại trong 6 tháng", còn "trôi xuống"
tương đương "im lặng". Đọc ma trận theo lăng kính này:

- Chỉ **1.3%** khách Engaged mua lại đủ để trụ lại nhóm; phần còn lại trôi xuống Cooling/Dormant.
- Tỷ lệ tái kích hoạt gần như bằng 0: Cooling hồi phục 1.0%, Dormant hồi 0.7%. Một khi khách nguội
  đi, gần như không quay lại.
- Dòng chảy gần như một chiều: Engaged → Cooling → Dormant, cực hiếm khi đảo ngược.

Đây là bằng chứng định lượng cho hiện tượng doanh nghiệp giữ được tổng số khách chủ yếu nhờ liên
tục thu hút khách mới chứ không phải nhờ giữ chân. Vì khoảng cách kỳ trùng ngưỡng như trên, phần
2.6 bổ sung một thước đo không phụ thuộc ngưỡng để có con số ổn định hơn.

### 2.4. Trọng số doanh thu theo luồng dịch chuyển

Ma trận đếm khách cho biết *bao nhiêu người* dịch chuyển, nhưng không phải mọi khách có giá trị
như nhau. Ở đây dùng monetary (tổng chi tiêu tích lũy của khách) làm trọng số: mỗi luồng dịch
chuyển được đo bằng tổng doanh thu mà nhóm khách đó đại diện. Câu hỏi chuyển từ "bao nhiêu khách
rời đi" sang "bao nhiêu giá trị đang rời đi".

In [4]:
# Ma trận tổng monetary theo luồng (thay vì đếm khách)
rev_matrix = pd.crosstab(
    merged['lifecycle_k1'], merged['lifecycle_k2'],
    values=merged['monetary'], aggfunc='sum'
).reindex(index=order, columns=order)

print("Tổng monetary theo luồng dịch chuyển (BRL):")
print(rev_matrix.round(0))

# % doanh thu mỗi luồng trên tổng toàn bộ khách có mặt cả 2 kỳ
total_rev = merged['monetary'].sum()
print(f"\nTổng giá trị nhóm khách xét: {total_rev:,.0f} BRL")
print("\n% doanh thu theo luồng:")
print((rev_matrix / total_rev * 100).round(1))

# Điểm nhấn: giá trị "chảy xuống Dormant" từ các nhóm còn sống
leak = rev_matrix.loc[['Engaged','Cooling'], 'Dormant'].sum()
print(f"\nDoanh thu từ khách Engaged/Cooling (kỳ 1) đã rơi xuống Dormant (kỳ 2): {leak:,.0f} BRL")
print(f"= {leak/total_rev*100:.1f}% tổng giá trị nhóm khách xét")

Error: No connection selected.

**Nhận xét:** Trong 6 tháng, khoảng 40% tổng giá trị của nhóm khách theo dõi đã chuyển từ trạng
thái còn hoạt động (Engaged/Cooling) sang Dormant. (Con số điểm này nhạy với ngưỡng churn; xem
kiểm định độ nhạy ở phần 2.5.)

Đáng chú ý hơn về mặt hành động là luồng **Engaged → Cooling** (cũng khoảng 40% giá trị): đây là
nhóm giá trị lớn vừa mới nguội, chưa mất hẳn nên còn khả năng giữ chân. Khác với phần đã chuyển
thẳng sang Dormant (khó thu hồi), nhóm Cooling là nhóm ưu tiên can thiệp rõ ràng nhất.

Kết nối các phần: survival analysis xác định thời điểm nên can thiệp (~175 ngày); migration cho
thấy nhóm cần can thiệp mang phần lớn giá trị và đang ở đúng thời điểm đó. Đây là đầu vào cho mô
phỏng business case ở phần 3.

### Trực quan hóa: heatmap ma trận chuyển tiếp

Hai heatmap thể hiện ma trận chuyển tiếp — bên trái theo số khách (% mỗi hàng), bên phải theo
doanh thu (% tổng). Màu càng đậm là luồng càng lớn. Đường chéo là khách giữ nguyên trạng thái;
phần dưới đường chéo là dịch chuyển xấu đi (nguội dần), phần trên là hồi phục.

In [5]:
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Heatmap % khách theo hàng
pct_cust = transition.div(transition.sum(axis=1), axis=0) * 100
sns.heatmap(pct_cust, annot=True, fmt='.1f', cmap='Oranges', ax=axes[0], cbar_kws={'label': '%'})
axes[0].set_title('Dịch chuyển theo SỐ KHÁCH (% mỗi hàng)')
axes[0].set_xlabel('Trạng thái kỳ 2'); axes[0].set_ylabel('Trạng thái kỳ 1')

# Heatmap % doanh thu
pct_rev = rev_matrix / total_rev * 100
sns.heatmap(pct_rev, annot=True, fmt='.1f', cmap='Reds', ax=axes[1], cbar_kws={'label': '% doanh thu'})
axes[1].set_title('Dịch chuyển theo DOANH THU (% tổng)')
axes[1].set_xlabel('Trạng thái kỳ 2'); axes[1].set_ylabel('Trạng thái kỳ 1')

plt.tight_layout()
plt.show()

Error: No connection selected.

### 2.5. Kiểm định độ nhạy của ngưỡng (Sensitivity Analysis)

Ngưỡng churn (P90 ≈ 288 ngày) là một phán đoán có căn cứ, không phải con số tối ưu tuyệt đối —
việc chọn P85 hay P95 phụ thuộc vào chi phí của hai loại sai lầm (can thiệp quá sớm gây lãng phí,
hay quá muộn nên mất khách). Để kiểm tra kết luận có phụ thuộc vào lựa chọn ngưỡng cụ thể không,
ta chạy lại phân tích migration với các ngưỡng "đã rời bỏ" khác nhau (P85, P90, P95) và xem tỷ lệ
giá trị chảy xuống Dormant có ổn định không. Nếu kết luận không đổi nhiều, phân tích là **bền vững** — không lệ thuộc vào một con số chọn tùy ý.

In [6]:
# Ngưỡng "đã rời bỏ" ứng với các phân vị khác nhau
for label, q in [('P85', 0.85), ('P90', 0.90), ('P95', 0.95)]:
    churn_thr = gap_clean.quantile(q)

    def stage_dyn(r, thr=churn_thr):
        if r <= 175: return 'Engaged'
        elif r <= thr: return 'Cooling'
        else: return 'Dormant'

    # Gán lại nhãn 2 kỳ với ngưỡng mới
    p1_s = p1.copy(); p2_s = p2.copy()
    p1_s['lc'] = p1_s['recency'].apply(stage_dyn)
    p2_s['lc'] = p2_s['recency'].apply(stage_dyn)

    m = p1_s[['customer_unique_id','lc']].merge(
        p2_s[['customer_unique_id','lc','monetary']], on='customer_unique_id', suffixes=('_k1','_k2'))

    # % giá trị từ nhóm còn sống (Engaged/Cooling kỳ 1) rơi xuống Dormant kỳ 2
    total = m['monetary'].sum()
    leak = m[(m['lc_k1'].isin(['Engaged','Cooling'])) & (m['lc_k2']=='Dormant')]['monetary'].sum()
    print(f"{label} (ngưỡng {churn_thr:.0f} ngày): {leak/total*100:.1f}% giá trị rơi xuống Dormant")

Error: No connection selected.

**Nhận xét:** Tỷ lệ giá trị chảy xuống Dormant dao động mạnh theo ngưỡng: 34.8% (P95) đến 50.8%
(P85) — biên độ gần 1.5 lần.

**Nguyên nhân**: do phần lớn khách chỉ mua một lần và phân bố recency
liên tục, tệp khách không có ranh giới tự nhiên giữa các trạng thái; mọi ngưỡng đều cắt ngang
vùng đông đúc, nên dịch ngưỡng nhỏ cũng làm nhiều khách đổi nhóm.

**Hệ quả**: con số phần trăm này phản ánh xu hướng (giá trị đang rò rỉ đáng kể) nhưng KHÔNG đủ ổn
định để dùng làm cơ sở tính toán tài chính chính xác. Cho các ước lượng định lượng (phần 3), nên
dùng thước đo không phụ thuộc ngưỡng — ví dụ tỷ lệ khách không phát sinh đơn nào trong kỳ 6 tháng.

### 2.6. Thước đo không phụ thuộc ngưỡng: tỷ lệ tái mua trong kỳ

Con số dịch chuyển ở phần trên phụ thuộc vào ngưỡng recency nên nhạy. Để có một thước đo bền vững
làm cơ sở định lượng, ta đo trực tiếp một sự kiện nhị phân không cần ngưỡng: trong số khách đã tồn
tại ở kỳ 1, bao nhiêu phần trăm **không phát sinh đơn hàng mới nào** trong 6 tháng giữa hai kỳ.
Đây là hành vi thực tế (có mua lại hay không), không phụ thuộc bất kỳ ranh giới phân loại nào.

In [7]:
k1_cutoff = pd.Timestamp('2018-02-28')
k2_cutoff = pd.Timestamp('2018-08-30')

# Khách phát sinh ÍT NHẤT 1 đơn mới trong cửa sổ (k1, k2]
repurchasers = set(
    df[(df['order_purchase_date'] > k1_cutoff) &
       (df['order_purchase_date'] <= k2_cutoff)]['customer_unique_id']
)

# Trong nhóm khách kỳ 1 (p1), ai có mua lại?
p1_repurchased = p1['customer_unique_id'].isin(repurchasers)

pct_no_rep = (~p1_repurchased).mean() * 100
print(f"Khách kỳ 1: {len(p1):,}")
print(f"% KHÔNG mua lại trong 6 tháng: {pct_no_rep:.1f}%")
print(f"% CÓ mua lại (tái kích hoạt):  {100-pct_no_rep:.1f}%")

# Theo giá trị: dùng monetary tích lũy đến kỳ 1 làm trọng số
val_no = p1.loc[~p1_repurchased, 'monetary'].sum()
val_total = p1['monetary'].sum()
print(f"\n% giá trị (kỳ 1) thuộc khách không mua lại: {val_no/val_total*100:.1f}%")

Error: No connection selected.

**Nhận xét:** Chỉ 1.2% khách tồn tại ở kỳ 1 phát sinh đơn hàng mới trong 6 tháng tiếp theo —
tỷ lệ tái kích hoạt tự nhiên cực thấp. Khác với chỉ số dịch chuyển theo ngưỡng, con số này là
hành vi nhị phân (có mua lại hay không) nên hoàn toàn ổn định, không phụ thuộc giả định phân loại.

Tỷ lệ theo giá trị (98.8%) trùng với tỷ lệ theo số khách (98.8%), cho thấy nhóm tái kích hoạt
không tập trung ở khách giá trị cao — giá trị phân bố tương đối đồng đều giữa nhóm quay lại và
không quay lại.

Lưu ý : con số 98.8% phản ánh khách *không tạo doanh thu mới* trong kỳ, không phải "mất
98.8% giá trị". Đây là thước đo tái mua (reactivation) trong
cửa sổ 6 tháng, tệp khách gần như
không tự tái kích hoạt nếu không có can thiệp.

### 2.7. Tổng kết phần dịch chuyển vòng đời

Phân tích migration cho ba kết luận chính:

1. **Dòng chảy gần như một chiều theo hướng nguội dần** (Engaged → Cooling → Dormant), tỷ lệ hồi
   phục ở mọi trạng thái chỉ khoảng 1% — tệp khách hàng gần như không tự quay lại nếu không có
   can thiệp.
2. **Nhóm Engaged → Cooling là nhóm ưu tiên can thiệp**: mang khoảng 40% giá trị, vừa mới nguội
   nên còn khả năng giữ chân, và đang ở đúng thời điểm can thiệp mà survival analysis chỉ ra.
3. **Chỉ 1.2% khách tái mua trong 6 tháng** (con số không phụ thuộc ngưỡng): bằng chứng ổn định
   cho thấy doanh nghiệp duy trì quy mô tệp khách chủ yếu nhờ thu hút khách mới, không phải giữ
   chân khách hiện có — và là baseline cho mô phỏng business case ở phần 3.

Về phương pháp: con số dịch chuyển theo trạng thái mang tính định hướng (độ lớn nhạy với ngưỡng),
trong khi tỷ lệ tái mua nhị phân cho con số ổn định để tính toán. Hai thước đo bổ sung nhau — một
mô tả cấu trúc dòng chảy, một cung cấp con số đáng tin.

## 3. Mô phỏng business case cho chiến dịch giữ chân

**Mục tiêu:** Ước lượng giá trị doanh thu có thể thu được nếu triển khai chiến dịch win-back cho
nhóm khách ưu tiên, và xác định điều kiện để chiến dịch có lãi. Đây là bước chuyển từ phân tích
sang khuyến nghị định lượng.

**Bản chất và giới hạn:** Đây là mô phỏng *trước* hành động dựa trên kịch bản, không phải dự báo
kết quả thực tế (việc đó cần A/B test — thiết kế ở phần 4). Mô hình được neo tối đa vào số liệu
thực; chỉ hai tham số là giả định, và đó cũng là hai tham số mà bất kỳ doanh nghiệp nào cũng phải
giả định trước khi chạy chiến dịch:

| Tham số | Nguồn | Giá trị |
| :-- | :-- | :-- |
| Số khách mục tiêu (nhóm Cooling) | Thực — đếm từ dữ liệu | tính ở dưới |
| Giá trị mỗi lần mua lại (AOV) | Thực — tính từ dữ liệu | tính ở dưới |
| Tỷ lệ tự quay lại của nhóm (baseline) | Thực — đo từ hành vi nhóm Cooling kỳ 1 | tính ở dưới |
| Tỷ lệ quay lại **tăng thêm** nhờ chiến dịch (uplift) | Giả định — chạy nhiều kịch bản | 1% / 3% / 5% |
| Chi phí mỗi khách | Giả định — tham số hóa | mặc định để minh họa |

**Nguyên tắc then chốt — trừ baseline:** chỉ phần tỷ lệ quay lại *vượt trên* mức tự nhiên mới được
tính là hiệu quả của chiến dịch. Nếu không trừ baseline, hiệu quả bị thổi phồng.

### 3.1. Xác định tham số từ dữ liệu

Nhóm mục tiêu là khách Cooling (đang nguội, còn khả năng giữ chân). Baseline được đo từ chính hành
vi thực của nhóm Cooling ở kỳ 1: trong số khách Cooling tại 28/02/2018, bao nhiêu phần trăm tự
phát sinh đơn mới trong 6 tháng tiếp theo mà không cần can thiệp.

In [8]:
# Số khách mục tiêu: nhóm Cooling ở kỳ hiện tại (kỳ 2)
N_target = (p2['lifecycle'] == 'Cooling').sum()

# AOV thực: doanh thu / số đơn
AOV = df['total_payment_value'].sum() / df['order_id'].nunique()

# Baseline: tỷ lệ tự quay lại THỰC của nhóm Cooling kỳ 1 trong 6 tháng
cooling_k1 = p1.loc[p1['lifecycle'] == 'Cooling', 'customer_unique_id']
baseline = cooling_k1.isin(repurchasers).mean()

print(f"Số khách mục tiêu (Cooling): {N_target:,}")
print(f"AOV: R$ {AOV:.2f}")
print(f"Tỷ lệ tự quay lại của nhóm Cooling (baseline): {baseline*100:.2f}%")

Error: No connection selected.

### 3.2. Mô phỏng doanh thu và ngưỡng hòa vốn

Với mỗi kịch bản uplift (tỷ lệ quay lại tăng thêm nhờ chiến dịch, tính trên mức baseline 1.08%),
ước lượng số khách quay lại thêm và doanh thu tương ứng. Thay vì giả định một mức chi phí cụ thể,
ta tính trực tiếp **chi phí tối đa mỗi khách để chiến dịch hòa vốn** — đây là ngưỡng hành động
được mà không cần giả định chi phí: chiến dịch chỉ có lãi nếu chi phí thực tế thấp hơn ngưỡng này.

In [9]:
scenarios = []
for uplift in [0.01, 0.03, 0.05]:
    extra_customers = N_target * uplift          # số khách quay lại THÊM nhờ chiến dịch
    extra_revenue = extra_customers * AOV        # doanh thu tăng thêm (gross)
    breakeven_cost = uplift * AOV                # chi phí tối đa/khách để hòa vốn
    scenarios.append({
        'Uplift': f"{uplift*100:.0f}%",
        'Tỷ lệ quay lại (từ baseline 1.08%)': f"1.08% → {(baseline+uplift)*100:.2f}%",
        'Khách quay lại thêm': f"{extra_customers:,.0f}",
        'Doanh thu tăng thêm (R$)': f"{extra_revenue:,.0f}",
        'Chi phí tối đa/khách để hòa vốn (R$)': f"{breakeven_cost:.2f}"
    })

pd.DataFrame(scenarios)

Error: No connection selected.

**Nhận xét:** Ngưỡng hòa vốn dao động từ R$1.60/khách (uplift 1%) đến R$8.00/khách (uplift 5%).
Đối chiếu với chi phí thực tế của các kênh tiếp cận:

- **Kênh chi phí thấp (email, push notification):** chi phí mỗi khách thường dưới R$1, thấp hơn cả
  ngưỡng hòa vốn của kịch bản uplift thấp nhất (R$1.60 ở uplift 1%). Win-back qua các kênh này gần
  như luôn có lãi ngay cả khi hiệu quả chiến dịch khiêm tốn — nên ưu tiên triển khai trước.
- **Kênh chi phí cao (voucher, giảm giá):** chi phí mỗi khách có thể lên hàng chục R$, chỉ hòa vốn
  nếu uplift đạt từ 3–5% trở lên. Cần kiểm chứng hiệu quả trước khi mở rộng quy mô.

**Khuyến nghị:** Với nhóm Cooling (25,114 khách), triển khai win-back bằng kênh chi phí thấp trước;
chỉ dùng ưu đãi tài chính đắt hơn nếu có bằng chứng uplift đủ cao.

**Giới hạn của ước lượng:** 

(1) Giá trị mỗi lần quay lại dùng AOV một đơn (R$159.86) — đây là ước
lượng thận trọng, giả định khách chỉ mua thêm một lần; nếu họ trở thành khách lặp lại, giá trị
thực cao hơn. 

(2) Uplift là giả định, chưa có bằng chứng thực nghiệm — phải được xác nhận bằng
A/B test (phần 4) trước khi tin.

### 3.3. Tổng kết business case

Với nhóm mục tiêu 25,114 khách Cooling, mô phỏng cho thấy chiến dịch win-back có thể mang lại từ
R$40,000 (uplift 1%) đến R$200,000 (uplift 5%) doanh thu tăng thêm. Điều kiện có lãi được xác định
qua ngưỡng hòa vốn: chi phí mỗi khách phải thấp hơn R$1.60 đến R$8.00 tùy mức uplift.

Kết luận thực tế: kênh chi phí thấp (email, push) gần như luôn có lãi cho nhóm này, trong khi ưu
đãi tài chính đắt tiền cần được kiểm chứng hiệu quả trước. Con số then chốt chưa xác định được là
uplift thực tế — đây chính là lý do cần một thí nghiệm có kiểm soát để đo, thay vì dựa vào giả
định. Phần 4 thiết kế thí nghiệm đó.

## 4. Thiết kế A/B test kiểm chứng chiến dịch win-back

**Mục tiêu:** Thiết kế một thí nghiệm có kiểm soát để đo hiệu quả thực tế của chiến
dịch win-back — cụ thể là uplift (tỷ lệ quay lại tăng thêm), tham số giả định duy nhất còn lại
trong business case ở phần 3.

**Bản chất:** Dữ liệu Olist kết thúc năm 2018 nên không thể chạy thí nghiệm thực; đây là bản thiết
kế phương pháp. Nó xác định cách thu thập bằng chứng để thay thế giả định uplift bằng số đo thực
trước khi mở rộng chiến dịch.

### 4.1. Giả thuyết và thiết kế nhóm

**Giả thuyết:**
- H₀ (giả thuyết không): chiến dịch win-back không làm thay đổi tỷ lệ quay lại của nhóm Cooling.
- H₁ (giả thuyết đối): chiến dịch làm tăng tỷ lệ quay lại so với không can thiệp.

**Thiết kế nhóm:**
- **Nhóm treatment:** nhận chiến dịch win-back (email/push).
- **Nhóm control:** không nhận gì.
- Chia ngẫu nhiên nhóm Cooling thành hai, tỷ lệ 50/50.

**Vì sao bắt buộc có nhóm control:** nhóm Cooling tự quay lại 1.08% ngay cả khi không can thiệp
(baseline đo ở phần 3). Nếu chỉ gửi chiến dịch cho tất cả rồi đếm số quay lại, không thể tách phần
do chiến dịch khỏi phần tự nhiên. Nhóm control đo chính mức tự nhiên đó trong cùng khoảng thời
gian, nên **uplift thực = tỷ lệ quay lại nhóm treatment − tỷ lệ nhóm control**.

**Vì sao chia ngẫu nhiên:** ngẫu nhiên hóa đảm bảo hai nhóm tương đương về mọi đặc điểm (giá trị,
recency, khu vực...) trừ việc có nhận chiến dịch hay không. Nhờ đó chênh lệch quan sát được có thể
quy cho chiến dịch, không phải do khác biệt sẵn có giữa hai nhóm.

**Chỉ số chính (primary metric):** tỷ lệ khách phát sinh ít nhất một đơn hàng trong vòng 60 ngày
kể từ khi chiến dịch gửi đi.

### 4.2. Tính cỡ mẫu (Power Analysis)

Câu hỏi then chốt: nhóm Cooling (25,114 khách, chia đôi thành ~12,557 mỗi nhóm) có đủ lớn để phát
hiện uplift đáng quan tâm không? Với baseline thấp (1.08%), phát hiện một uplift nhỏ cần cỡ mẫu
lớn. Ta tính cỡ mẫu tối thiểu mỗi nhóm cần có để phát hiện các mức uplift khác nhau, ở mức ý nghĩa
5% và power 80% (chuẩn thông dụng).

In [10]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

baseline_p = 0.0108          # tỷ lệ quay lại nhóm control (baseline Cooling)
available_per_group = N_target / 2   # số khách mỗi nhóm nếu chia đôi

print(f"Số khách mỗi nhóm nếu chia 50/50: {available_per_group:,.0f}\n")

analysis = NormalIndPower()
for uplift in [0.005, 0.01, 0.02, 0.03]:
    treat_p = baseline_p + uplift
    effect = proportion_effectsize(treat_p, baseline_p)      # Cohen's h
    n_needed = analysis.solve_power(effect_size=effect, alpha=0.05, power=0.80, alternative='larger')
    verdict = "ĐỦ" if n_needed <= available_per_group else "KHÔNG đủ"
    print(f"Uplift {uplift*100:.1f}% (1.08% → {treat_p*100:.2f}%): "
          f"cần {n_needed:,.0f}/nhóm → {verdict}")

Error: No connection selected.

**Nhận xét:** Nhóm Cooling chia đôi được 12,557 khách mỗi nhánh — đủ lớn để phát hiện uplift từ
0.5% trở lên với độ tin cậy chuẩn (α=5%, power=80%). Với các mức uplift thực tế hơn (1–3%), số
khách cần thiết chỉ vài trăm đến gần 2,000 mỗi nhóm, ít hơn nhiều so với số khách đang có.

Hệ quả: số lượng khách không phải vấn đề. Vì có nhiều hơn hẳn mức cần, có thể chọn kiểm chứng cả
những uplift rất nhỏ, hoặc chia nhóm nhỏ hơn để thử nhiều kiểu chiến dịch cùng lúc (ví dụ so email
với push, hoặc so các mức ưu đãi khác nhau).

Lưu ý về con số thật: baseline 1.08% nghĩa là mỗi nhóm 12,557 khách chỉ có khoảng 135 người tự
quay lại; uplift 1% tương ứng thêm khoảng 125 người. Đây là những con số nhỏ, nên thí nghiệm cần
chạy đủ dài (cửa sổ 60 ngày) để gom đủ số người quay lại và theo dõi cẩn thận.

### 4.3. Tiêu chí quyết định và cạm bẫy cần tránh

**Phương pháp kiểm định:** so sánh tỷ lệ quay lại hai nhóm bằng two-proportion z-test (một phía).
Tuyên bố chiến dịch có hiệu quả nếu p-value < 0.05.

**Ý nghĩa thống kê chưa đủ — cần cả ý nghĩa thực tiễn:** một kết quả có p-value < 0.05 chỉ khẳng
định uplift khác 0, chưa khẳng định nó *đáng tiền*. Uplift còn phải vượt ngưỡng hòa vốn từ business
case (phần 3): ví dụ nếu dùng ưu đãi chi phí R$4.80/khách, uplift phải đạt ít nhất 3% mới có lãi.
Quyết định mở rộng chiến dịch cần thỏa **cả hai**: có ý nghĩa thống kê *và* vượt ngưỡng hòa vốn.

**Cạm bẫy cần tránh:**
- **Dừng sớm khi thấy kết quả đẹp:** kiểm tra kết quả liên tục rồi dừng ngay khi thấy
  p-value < 0.05 làm tăng mạnh tỷ lệ dương tính giả. Phải cố định thời gian chạy trước khi bắt đầu.
- **Hiệu ứng mới lạ:** khách có thể phản ứng với chiến dịch chỉ vì nó mới, khiến
  uplift ban đầu cao hơn mức bền vững. Cần chạy đủ dài để đánh giá hiệu quả ổn định.
- **Thời gian chạy đủ dài:** cửa sổ đo phải đủ để khách có cơ hội quay lại theo hành vi
  mua thực tế của nhóm.
- **Đồng thời về thời gian:** control và treatment phải chạy cùng khoảng thời gian để loại ảnh
  hưởng mùa vụ.

### 4.4. Tổng kết thiết kế A/B test

Thí nghiệm được thiết kế để đo hiệu quả thực của chiến dịch win-back, thay thế giả định uplift ở
phần 3 bằng số đo có kiểm soát. Nhóm Cooling đủ lớn để chạy thí nghiệm này với độ tin cậy chuẩn.

Điểm mấu chốt: quyết định mở rộng chiến dịch cần thỏa cả hai điều kiện — uplift có ý nghĩa thống kê
(p-value < 0.05) và vượt ngưỡng hòa vốn từ business case (phần 3). Đây là điểm nối giữa hai phần:
business case đề xuất điều kiện cần đạt, A/B test đo xem có đạt hay không, và kết quả đo lại cập
nhật business case cho lần triển khai tiếp theo.

## Kết luận phân tích nâng cao

Phân khúc RFM ở phân tích trước là một ảnh chụp tại một thời điểm: nó cho biết mỗi khách hiện thuộc
nhóm nào, nhưng không cho biết hai điều — *khi nào* một khách được coi là đã rời bỏ, và khách
*dịch chuyển* giữa các nhóm ra sao theo thời gian. Notebook này bổ sung đúng hai khía cạnh đó:
thời gian (survival analysis) và động lực dịch chuyển (migration).

**Các phát hiện chính:**

1. **Tỷ lệ mua lại thực thấp hơn ước tính ban đầu.** Sau khi loại các đơn bị tách trong cùng một
   lần mua (kiểm chứng ở mức timestamp: chênh nhau trung bình 1 giây), tỷ lệ khách mua lại thực chỉ
   2.16% thay vì 3.0% — nghĩa là 97.84% khách chỉ mua một lần.

2. **Ngưỡng rời bỏ nên dựa trên dữ liệu, không dùng quy ước.** Quy ước 90 ngày phân loại nhầm khoảng
   45% khách còn khả năng quay lại là đã mất. Phân tích đề xuất hai ngưỡng: can thiệp ở ~175 ngày,
   coi là đã rời bỏ ở ~288 ngày.

3. **Tệp khách gần như không tự tái kích hoạt.** Chỉ 1.2% khách phát sinh đơn mới trong 6 tháng
   (con số ổn định, không phụ thuộc ngưỡng). Doanh nghiệp duy trì quy mô chủ yếu nhờ thu hút khách
   mới chứ không phải giữ chân khách hiện có.

4. **Nhóm Cooling là mục tiêu can thiệp rõ ràng nhất** — mang phần lớn giá trị và đang ở đúng thời
   điểm can thiệp. Win-back qua kênh chi phí thấp gần như luôn có lãi (ngưỡng hòa vốn R$1.60/khách),
   và nhóm đủ lớn để kiểm chứng hiệu quả bằng A/B test.

**Mạch phân tích:** từ mô tả (khách thuộc nhóm nào — phân khúc RFM) sang chẩn đoán (khi nào rời bỏ,
dịch chuyển ra sao — survival và migration) sang khuyến nghị có điều kiện (chiến dịch nào đáng làm,
kiểm chứng thế nào — business case và A/B test).